
# **LTX2.3 for Text & Image to Video+Audio with ComfyUI**

---

- If using the free T4 GPU with the default workflow models and settings, you can generate up to 5 seconds of a 480p video in around 20 minutes in text to video mode, and 3 seconds of a 480x832 video in around 17 minutes in image to video mode.
- You can generate up to 20 seconds of a 720p video on the L4 GPU in less than 15 minutes.
- Run the cell below to get a link (e.g. https://localhost:8188/) which you can use to launch the comfyUI interface. Then drag into the interface a workflow or app downloaded from the workflows link below.
- Workflows: https://github.com/Isi-dev/Google-Colab_Notebooks/tree/main/ComfyUI/ComfyUI_LTX2_3
- **Note**: Also load an image for text to video to avoid errors.
- Models source: (1) https://huggingface.co/Lightricks/LTX-2.3/tree/main (2) https://huggingface.co/unsloth/LTX-2.3-GGUF/tree/main (3) https://huggingface.co/unsloth/gemma-3-12b-it-qat-GGUF/tree/main
- Github project page: https://github.com/Lightricks/LTX-2
- Notebook source: https://github.com/Isi-dev/Google-Colab_Notebooks
- Premium notebooks: https://isinse.gumroad.com/





In [2]:
# @title {"single-column":true}
# @markdown # 💥Prepare Environment

!pip install torch torchvision

offload_models_for_low_VRAM = True # @param {type:"boolean"}
include_manager = False # @param {type:"boolean"}
include_ltxNodes = True # @param {type:"boolean"}
include_rgthree = True # @param {type:"boolean"}
include_Easy_Use = True # @param {type:"boolean"}
include_VideoHelperSuite = True # @param {type:"boolean"}
include_MelBandRoFormer = True # @param {type:"boolean"}

%cd /content
from IPython.display import clear_output
clear_output()
!pip install -q torchsde einops diffusers accelerate
!pip install av spandrel albumentations onnx opencv-python onnxruntime
!git clone https://github.com/comfyanonymous/ComfyUI
!pip install -r /content/ComfyUI/requirements.txt
clear_output()


%cd /content/ComfyUI/custom_nodes
if include_manager:
    !git clone https://github.com/Comfy-Org/ComfyUI-Manager

!git clone https://github.com/kijai/ComfyUI-KJNodes
!git clone https://github.com/city96/ComfyUI-GGUF
if include_ltxNodes:
    !git clone https://github.com/Lightricks/ComfyUI-LTXVideo/
if include_rgthree:
    !git clone https://github.com/rgthree/rgthree-comfy.git
if include_Easy_Use:
    !git clone https://github.com/yolain/ComfyUI-Easy-Use
if include_VideoHelperSuite:
    !git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite
if include_MelBandRoFormer:
    !git clone https://github.com/kijai/ComfyUI-MelBandRoFormer
%cd /content/ComfyUI/custom_nodes/ComfyUI-KJNodes
!pip install -r requirements.txt
%cd /content/ComfyUI/custom_nodes/ComfyUI-GGUF
!pip install -r requirements.txt
if include_Easy_Use:
    %cd /content/ComfyUI/custom_nodes/ComfyUI-Easy-Use
    !pip install -r requirements.txt
if include_VideoHelperSuite:
    %cd /content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite
    !pip install -r requirements.txt
if include_MelBandRoFormer:
    %cd /content/ComfyUI/custom_nodes/ComfyUI-MelBandRoFormer
    !pip install -r requirements.txt
if include_manager:
    %cd /content/ComfyUI/custom_nodes/ComfyUI-Manager
    !pip install -r requirements.txt

clear_output()


%cd /content/ComfyUI


import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import subprocess
import sys
from pathlib import Path

def install_apt_packages():
    packages = ['aria2']

    try:
        subprocess.run(
            ['apt-get', '-y', 'install', '-qq'] + packages,
            check=True,
            capture_output=True
        )
        print("✓ apt packages installed")
    except subprocess.CalledProcessError as e:
        print(f"✗ Error installing apt packages: {e.stderr.decode().strip() or 'Unknown error'}")


print("Installing apt packages...")
install_apt_packages()

def download_with_aria2c(link, folder="/content/ComfyUI/models/loras"):
    import os
    filename = link.split("/")[-1]
    command = f"aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {link} -d {folder} -o {filename}"
    print("Executing download command:")
    print(command)
    os.makedirs(folder, exist_ok=True)
    get_ipython().system(command)
    return filename

def download_civitai_model(civitai_link, civitai_token, folder="/content/ComfyUI/models/loras"):
    import os
    import time
    os.makedirs(folder, exist_ok=True)
    try:
        model_id = civitai_link.split("/models/")[1].split("?")[0]
    except IndexError:
        raise ValueError("Invalid Civitai URL format.")
    civitai_url = f"https://civitai.com/api/download/models/{model_id}?type=Model&format=SafeTensor"
    if civitai_token:
        civitai_url += f"&token={civitai_token}"
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    filename = f"model_{timestamp}.safetensors"
    full_path = os.path.join(folder, filename)
    download_command = f"wget --max-redirect=10 --show-progress \"{civitai_url}\" -O \"{full_path}\""
    print("Downloading from Civitai...")
    os.system(download_command)
    return filename

def download_lora(link, folder="/content/ComfyUI/models/loras", civitai_token=None):
    if "civitai.com" in link.lower():
        return download_civitai_model(link, civitai_token, folder)
    else:
        return download_with_aria2c(link, folder)

def model_download(url, dest_dir, filename=None, silent=True):
    try:
        Path(dest_dir).mkdir(parents=True, exist_ok=True)
        if filename is None:
            filename = url.split('/')[-1].split('?')[0]
        cmd = ['aria2c', '--console-log-level=error', '-c', '-x', '16', '-s', '16', '-k', '1M', '-d', dest_dir, '-o', filename, url]
        if silent:
            cmd.extend(['--summary-interval=0', '--quiet'])
            print(f"Downloading {filename}...", end=' ', flush=True)
        subprocess.run(cmd, check=True, capture_output=True, text=True)
        if silent: print("Done!")
        return filename
    except Exception as e:
        print(f"\nError: {str(e)}")
        return False

# Model Configuration
ltx_model = "https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/ltx-2.3-22b-dev-Q4_K_M.gguf" # @param {"type":"string"}
dit_model=model_download(ltx_model, "/content/ComfyUI/models/unet")

text_encoder_link = "https://huggingface.co/Comfy-Org/ltx-2/resolve/main/split_files/text_encoders/gemma_3_12B_it_fp8_scaled.safetensors" # @param {"type":"string"}
text_encoder_model = model_download(text_encoder_link, "/content/ComfyUI/models/text_encoders")

text_encoder2_link = "https://huggingface.co/unsloth/gemma-3-12b-it-qat-GGUF/resolve/main/mmproj-BF16.gguf" # @param {"type":"string"}
text_encoder2_model = model_download(text_encoder2_link, "/content/ComfyUI/models/text_encoders")

text_encoder3_link = "https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/text_encoders/ltx-2.3-22b-dev_embeddings_connectors.safetensors" # @param {"type":"string"}
text_encoder3_model = model_download(text_encoder3_link, "/content/ComfyUI/models/text_encoders")

vae_link = "https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/vae/ltx-2.3-22b-dev_video_vae.safetensors" # @param {"type":"string"}
vae_model = model_download(vae_link, "/content/ComfyUI/models/vae")

vae_audio_link = "https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/vae/ltx-2.3-22b-dev_audio_vae.safetensors" # @param {"type":"string"}
vae_audio_model = model_download(vae_audio_link, "/content/ComfyUI/models/vae")

upscaler_link = "https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-spatial-upscaler-x2-1.0.safetensors" # @param {"type":"string"}
upscaler_model = model_download(upscaler_link, "/content/ComfyUI/models/latent_upscale_models")

if include_MelBandRoFormer:
    model_download("https://huggingface.co/Kijai/MelBandRoFormer_comfy/resolve/main/MelBandRoformer_fp16.safetensors", "/content/ComfyUI/models/diffusion_models")

model_download("https://huggingface.co/Kijai/LTX2.3_comfy/resolve/main/vae/taeltx2_3.safetensors", "/content/ComfyUI/models/vae")

# LoRA Configuration
download_loRA1 = True # @param {type:"boolean"}
lora1_download_url = "https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-22b-distilled-lora-384.safetensors"
if download_loRA1: lora1 = download_lora(lora1_download_url)

# Interface Settings - Updated to use_interface_in_cell for security
use_cloudflare = False  # @param {type:"boolean"}
use_interface_in_cell = True  # @param {type:"boolean"}
use_ngrok = False  # @param {type:"boolean"}
NGROK_AUTH_TOKEN = ""  # @param {type:"string"}

%cd /content/ComfyUI

# ---------------- CLOUDFLARE ----------------
if use_cloudflare:
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb >/dev/null 2>&1
    import subprocess, threading, time, socket
    from IPython.display import clear_output
    def cloudflare_thread(port):
        while True:
            time.sleep(0.5)
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            if sock.connect_ex(('127.0.0.1', port)) == 0: break
            sock.close()
        p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"], stderr=subprocess.PIPE)
        for line in p.stderr:
            if "trycloudflare.com " in line.decode():
                clear_output(); print("✅ ComfyUI is ready!")
                print("🌐 Access it here:", line.decode()[line.decode().find("http"):])
                break
    threading.Thread(target=cloudflare_thread, daemon=True, args=(8188,)).start()
    !python main.py --cache-none --dont-print-server

# ---------------- IN-CELL INTERFACE ----------------
elif use_interface_in_cell:
    import threading, time, socket
    from google.colab import output
    from IPython.display import clear_output
    def iframe_thread(port):
        while True:
            time.sleep(0.5)
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            if sock.connect_ex(('127.0.0.1', port)) == 0: break
            sock.close()
        clear_output()
        output.serve_kernel_port_as_iframe(port, height=1024)
        print("✅ ComfyUI loaded inside the notebook.")
        output.serve_kernel_port_as_window(port)
    threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
    !python main.py --cache-none --dont-print-server

# ---------------- DEFAULT ----------------
else:
    import socket, time, threading
    from google.colab import output
    from IPython.display import clear_output
    def colab_link_thread(port):
        while True:
            time.sleep(0.5)
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            if sock.connect_ex(('127.0.0.1', port)) == 0: break
            sock.close()
        clear_output(); print("✅ ComfyUI is ready!")
        output.serve_kernel_port_as_window(port)
    threading.Thread(target=colab_link_thread, daemon=True, args=(8188,)).start()
    !python main.py --cache-none --dont-print-server

<IPython.core.display.Javascript object>

✅ ComfyUI loaded inside the notebook.
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

FETCH ComfyRegistry Data: 5/145
FETCH ComfyRegistry Data: 10/145
FETCH ComfyRegistry Data: 15/145
FETCH ComfyRegistry Data: 20/145
FETCH ComfyRegistry Data: 25/145
FETCH ComfyRegistry Data: 30/145
FETCH ComfyRegistry Data: 35/145
FETCH ComfyRegistry Data: 40/145
FETCH ComfyRegistry Data: 45/145
FETCH ComfyRegistry Data: 50/145
FETCH ComfyRegistry Data: 55/145
FETCH ComfyRegistry Data: 60/145
FETCH ComfyRegistry Data: 65/145
FETCH ComfyRegistry Data: 70/145
FETCH ComfyRegistry Data: 75/145
FETCH ComfyRegistry Data: 80/145
FETCH ComfyRegistry Data: 85/145
FETCH ComfyRegistry Data: 90/145
FETCH ComfyRegistry Data: 95/145
FETCH ComfyRegistry Data: 100/145
FETCH ComfyRegistry Data: 105/145
FETCH ComfyRegistry Data: 110/145
FETCH ComfyRegistry Data: 115/145
FETCH ComfyRegistry Data: 120/145
FETCH ComfyRegistry Data: 125/145
FETCH ComfyRegistry Data: 130/145
FETCH ComfyRegistry Data: 135/145
FETCH ComfyRegistry Data: 140/145
FETCH ComfyRegistry Data: 145/145
FETCH ComfyRegistry Data [DONE]
[C